# 3. Cohort key: confirm the SDH, set time zero, pick the index episode

**What it does**
- **Confirms the SDH on imaging:** an episode is confirmed when any of its CTs shows an SDH (NLP).
- **Index CT:** the first CT showing the SDH (its NLP features become the CT variables in the baseline).
- **Time zero:** ER triage at the first hospital (or admission) for patients who came in with the SDH;
  the first CT showing it for SDH that **developed in hospital**.
- **Index episode:** each patient's first confirmed episode (or their first episode, flagged, if none is).
- Gives every patient a study ID, `row_id` (SDH00001 ...), the key that links all later files.

If the NLP has not been run, it still works (every CT counts as showing the SDH) and says so.

**Output:** `patient_level/cohort_key.csv` (one row per patient); aggregate `step2_counts.csv`.
**Time:** under a minute.

> **Privacy:** everything this notebook *displays* comes from `aggregate/` (counts with cells under 6 suppressed). Files in `patient_level/` stay on GEMINI: don't print their rows, and never paste them outside GEMINI.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# To write the output somewhere else, uncomment and edit BEFORE importing sdh_config:
# os.environ["SDH_OUTPUT_DIR"] = str(Path.home() / "sdh_phenotype_output")

PHENO = Path.cwd().parent              # the phenotype/ folder (this notebook is in phenotype/notebooks/)
sys.path.insert(0, str(PHENO))
import sdh_config as cfg
import sdh_io as io
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 220)
AGG = cfg.AGGREGATE_DIR


def show(name, rows=None, note=None):
    """Display one aggregate (cell-suppressed) table from the output folder."""
    path = AGG / name
    if not path.exists():
        print(f"{name}: not there yet (run the step above first)")
        return None
    df = pd.read_csv(path)
    display(Markdown(f"**{name}**" + (f" - {note}" if note else "")))
    display(df.head(rows) if rows else df)
    return df


def share(names, rows=80):
    """Print aggregate tables as text to paste to Claude / your team (counts only, small cells suppressed)."""
    for name in names:
        path = AGG / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"===== {name} ({len(df)} rows{', first ' + str(rows) if len(df) > rows else ''}) =====")
            print(df.head(rows).to_csv(index=False))
        else:
            print(f"===== {name}: not found =====")

## Run

In [ ]:
%run ../02_cohort_key.py

## Results

In [ ]:
show("step2_counts.csv");

**Look for:**
- `index episode: first confirmed episode` should be most patients; the *none confirmed* group is
  mostly patients with no CT report or whose CTs never showed an SDH.
- `time zero from`: mostly `er_triage`; `diagnosis_ct` = in-hospital onset; `missing (no CT)` =
  in-hospital onset without a CT report (they cannot get a time zero).

In [ ]:
# timing summaries (aggregate only)
k = io.read_patient_level("cohort_key.csv")
def q(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return pd.Series({"patients": io.suppress(len(s)), "median": round(s.median(), 1),
                      "IQR": f"{s.quantile(.25):.1f} to {s.quantile(.75):.1f}"}) if len(s) >= cfg.SMALL_CELL else None
display(pd.DataFrame({
    "hours: time zero -> index CT": q(k.hours_t0_to_index_ct),
    "hours: admission -> SDH diagnosis (in-hospital onset)": q(k.hours_admission_to_sdh_diagnosis),
}).T)

**Look for:** index CT usually within a few hours of arrival; in-hospital onset typically days after admission.

## Share

In [ ]:
share(["step2_counts.csv"])